<a href="https://colab.research.google.com/github/SushrutReddy/CRUD/blob/main/DL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Question 1

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense, TimeDistributed
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split


In [ ]:
# ------------------ LOAD DATA ------------------ #
def load_data(file_path, max_samples=10000):
    input_texts, target_texts = [], []

    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

        print("🔍 File preview:")
        print("\n".join(lines[:5]))  # Show first 5 lines

        for line in lines[:max_samples]:
            parts = line.strip().split("\t")
            if len(parts) >= 2:
                input_text, target_text = parts[0], parts[1]
                input_texts.append(input_text)
                target_texts.append("\t" + target_text + "\n")  # Add start/end tokens

    return input_texts, target_texts

# Upload the dataset file when prompted
from google.colab import files
uploaded = files.upload()
file_path = next(iter(uploaded))

# Load and preview data
input_texts, target_texts = load_data(file_path)

print("✅ Samples loaded:", len(input_texts))
if len(input_texts) > 0:
    print("🔤 Example input:", input_texts[0])
    print("🔡 Example target:", target_texts[0])
else:
    print("⚠️ No valid data found. Please check the file format.")


Saving hi.translit.sampled.train.tsv to hi.translit.sampled.train.tsv
🔍 File preview:
अं	an	3

अंकगणित	ankganit	3

अंकल	uncle	4

अंकुर	ankur	4

अंकुरण	ankuran	3

✅ Samples loaded: 10000
🔤 Example input: अं
🔡 Example target: 	an



In [ ]:
# ------------------ TOKENIZATION ------------------ #
input_tokenizer = Tokenizer(char_level=True)
input_tokenizer.fit_on_texts(input_texts)
input_sequences = input_tokenizer.texts_to_sequences(input_texts)
input_vocab_size = len(input_tokenizer.word_index) + 1

target_tokenizer = Tokenizer(char_level=True)
target_tokenizer.fit_on_texts(target_texts)
target_sequences = target_tokenizer.texts_to_sequences(target_texts)
target_vocab_size = len(target_tokenizer.word_index) + 1


In [ ]:
# ------------------ PADDING ------------------ #
max_encoder_seq_length = max(len(seq) for seq in input_sequences)
max_decoder_seq_length = max(len(seq) for seq in target_sequences)

encoder_input_data = pad_sequences(input_sequences, maxlen=max_encoder_seq_length, padding='post')
decoder_input_data = pad_sequences(target_sequences, maxlen=max_decoder_seq_length, padding='post')

# Shift decoder_target_data by 1 timestep
decoder_target_data = np.zeros_like(decoder_input_data)
decoder_target_data[:, :-1] = decoder_input_data[:, 1:]

# ------------------ SPLIT ------------------ #
x_train, x_val, y_train, y_val, dec_train, dec_val = train_test_split(
    encoder_input_data, decoder_target_data, decoder_input_data, test_size=0.2, random_state=42
)


In [ ]:
# ------------------ BUILD MODEL ------------------ #
embedding_dim = 50
hidden_units = 256

# Encoder
encoder_inputs = Input(shape=(None,))
enc_emb = Embedding(input_vocab_size, embedding_dim)(encoder_inputs)
enc_outputs, state_h, state_c = LSTM(hidden_units, return_state=True)(enc_emb)
encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(None,))
dec_emb = Embedding(target_vocab_size, embedding_dim)(decoder_inputs)
decoder_lstm = LSTM(hidden_units, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = TimeDistributed(Dense(target_vocab_size, activation='softmax'))
decoder_outputs = decoder_dense(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 50)  │      3,100 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 50)  │      1,450 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 256),     │    314,368 │ embedding[0][0]   │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │    314,368 │ embedding_1[0][0… │
│                     │ 256), (None,      │            │ lstm[0][1],       │
│                     │ 256), (None,      │            │ lstm[0][2]        │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed    │ (None, None, 29)  │      7,453 │ lstm_1[0][0]      │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 640,739 (2.44 MB)

 Trainable params: 640,739 (2.44 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# ------------------ TRAIN MODEL ------------------ #
model.fit(
    [x_train, dec_train],
    np.expand_dims(y_train, -1),
    batch_size=64,
    epochs=15,
    validation_data=([x_val, dec_val], np.expand_dims(y_val, -1))
)


Epoch 1/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 29s 228ms/step - accuracy: 0.7106 - loss: 1.0120 - val_accuracy: 0.7326 - val_loss: 0.9063
Epoch 2/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 25s 199ms/step - accuracy: 0.7361 - loss: 0.8969 - val_accuracy: 0.7452 - val_loss: 0.8626
Epoch 3/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 42s 211ms/step - accuracy: 0.7530 - loss: 0.8445 - val_accuracy: 0.7581 - val_loss: 0.8135
Epoch 4/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 41s 213ms/step - accuracy: 0.7629 - loss: 0.8036 - val_accuracy: 0.7703 - val_loss: 0.7791
Epoch 5/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 44s 237ms/step - accuracy: 0.7724 - loss: 0.7733 - val_accuracy: 0.7741 - val_loss: 0.7574
Epoch 6/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 38s 212ms/step - accuracy: 0.7772 - loss: 0.7539 - val_accuracy: 0.7800 - val_loss: 0.7349
Epoch 7/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 41s 211ms/step - accuracy: 0.7843 - loss: 0.7259 - val_accuracy: 0.7886 - val_loss: 0.7027
Epoch 8/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 40s 205ms/step - accuracy: 0.7894 - loss: 0

In [ ]:
# ------------------ INFERENCE MODELS ------------------ #

# Encoder model
encoder_model = Model(encoder_inputs, encoder_states)

# Decoder Inference
decoder_state_input_h = Input(shape=(hidden_units,))
decoder_state_input_c = Input(shape=(hidden_units,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

# Define the decoder embedding layer
decoder_embedding = Embedding(target_vocab_size, embedding_dim)

decoder_input_single = Input(shape=(1,))
decoder_input_embedded = decoder_embedding(decoder_input_single)

decoder_lstm = LSTM(hidden_units, return_sequences=True, return_state=True)
decoder_outputs, state_h, state_c = decoder_lstm(
    decoder_input_embedded, initial_state=decoder_states_inputs
)
decoder_states = [state_h, state_c]

# Output layer for the decoder
decoder_dense = TimeDistributed(Dense(target_vocab_size, activation='softmax'))
decoder_outputs = decoder_dense(decoder_outputs)

# Create the decoder model
decoder_model = Model([decoder_input_single] + decoder_states_inputs, [decoder_outputs] + decoder_states)

# ------------------ DECODE FUNCTION ------------------ #
def decode_sequence(input_seq):
    # Encode the input as state vectors
    states_value = encoder_model.predict(input_seq)

    # Start with the start token
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = target_tokenizer.word_index['\t']

    stop_condition = False
    decoded_sentence = ''

    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)

        # Sample the token with the highest probability
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = reverse_target_char_index.get(sampled_token_index, '')

        decoded_sentence += sampled_char

        # Stop if we reach the end token or exceed max length
        if sampled_char == '\n' or len(decoded_sentence) > max_decoder_seq_length:
            stop_condition = True

        # Update the target sequence and states for the next iteration
        target_seq[0, 0] = sampled_token_index
        states_value = [h, c]

    return decoded_sentence.strip()

# ------------------ EVALUATION ------------------ #
correct = 0
total = len(test_inputs)
batch_size = 64

for i in range(0, total, batch_size):
    batch_input_seq = encoder_test_data[i:i + batch_size]
    batch_predicted = []

    for j in range(len(batch_input_seq)):
        input_seq = batch_input_seq[j:j + 1]
        predicted = decode_sequence(input_seq)
        batch_predicted.append(predicted)

    # Debugging: Print out actual vs predicted outputs
    for j, predicted in enumerate(batch_predicted):
        actual = test_targets[i + j].strip().lower()
        pred_clean = predicted.strip().lower()

        # Print input, actual, and predicted values for first few samples
        if i + j < 5:
            print(f"📝 Input: {test_inputs[i + j]}")
            print(f"✅ Actual: {actual}")
            print(f"🤖 Predicted: {pred_clean}\n")

        if pred_clean == actual:
            correct += 1

accuracy = correct / total
print(f"\n✅ Test Accuracy (on {total} samples): {accuracy * 100:.2f}%")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 207ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 220ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━

In [2]:
from google.colab import files
files.upload()  # Upload kaggle.json when prompted

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


Saving kaggle.json to kaggle.json


In [4]:
!kaggle datasets download -d paultimothymooney/poetry
!unzip poetry.zip

Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/poetry
License(s): CC0-1.0
Archive:  poetry.zip
  inflating: Kanye_West.txt          
  inflating: Lil_Wayne.txt           
  inflating: adele.txt               
  inflating: al-green.txt            
  inflating: alicia-keys.txt         
  inflating: amy-winehouse.txt       
  inflating: beatles.txt             
  inflating: bieber.txt              
  inflating: bjork.txt               
  inflating: blink-182.txt           
  inflating: bob-dylan.txt           
  inflating: bob-marley.txt          
  inflating: britney-spears.txt      
  inflating: bruce-springsteen.txt   
  inflating: bruno-mars.txt          
  inflating: cake.txt                
  inflating: dickinson.txt           
  inflating: disney.txt              
  inflating: dj-khaled.txt           
  inflating: dolly-parton.txt        
  inflating: dr-seuss.txt            
  inflating: drake.txt               
  inflating: eminem.txt              
  inflating: ja

In [5]:
!pip install transformers datasets accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [7]:
# Path to the drake.txt file
drake_file_path = './drake.txt'

# Read the file to confirm it's loaded
with open(drake_file_path, 'r', encoding='utf-8') as file:
    drake_lyrics = file.read()

print(drake_lyrics[:500])  # Display the first 500 characters of the lyrics to verify


[Hook]
I've been down so long, it look like up to me
They look up to me
I got fake people showin' fake love to me
Straight up to my face, straight up to my face
I've been down so long, it look like up to me
They look up to me
I got fake people showin' fake love to me
Straight up to my face, straight up to my face [Verse 1]
Somethin' ain't right when we talkin'
Somethin' ain't right when we talkin'
Look like you hidin' your problems
Really you never was solid
No, you can't "son" me
You won't neve


In [8]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Load the GPT-2 tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained("gpt2")
model.resize_token_embeddings(len(tokenizer))


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Embedding(50257, 768)

In [9]:
from transformers import TextDataset, DataCollatorForLanguageModeling

# Function to load the dataset (drake.txt) and create a text dataset
def load_dataset(file_path, tokenizer, block_size=128):
    return TextDataset(
        tokenizer=tokenizer,
        file_path=file_path,
        block_size=block_size
    )

# Function to get the data collator for language modeling
def get_data_collator(tokenizer):
    return DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    )

# Load the Drake dataset
dataset = load_dataset(drake_file_path, tokenizer)
data_collator = get_data_collator(tokenizer)


/usr/local/lib/python3.11/dist-packages/transformers/data/datasets/language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(


In [12]:
from transformers import Trainer, TrainingArguments
import torch

training_args = TrainingArguments(
    output_dir="./gpt2-drake-lyrics",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    prediction_loss_only=True,
    logging_steps=100,
    fp16=torch.cuda.is_available(),  # Enable mixed precision if GPU is available
    report_to='none',  # Disable W&B logging
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator,
)


In [13]:
trainer.train()
trainer.save_model("./gpt2-drake-lyrics")
tokenizer.save_pretrained("./gpt2-drake-lyrics")


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,3.587300
200,3.344600
300,3.140200
400,3.048700
500,2.854600
600,2.805600


('./gpt2-drake-lyrics/tokenizer_config.json',
 './gpt2-drake-lyrics/special_tokens_map.json',
 './gpt2-drake-lyrics/vocab.json',
 './gpt2-drake-lyrics/merges.txt',
 './gpt2-drake-lyrics/added_tokens.json')

In [14]:
from transformers import pipeline

# Load the trained model and tokenizer for generation
generator = pipeline("text-generation", model="./gpt2-drake-lyrics", tokenizer="./gpt2-drake-lyrics")

# Set your prompt
prompt = "Started from the bottom, now we're here"
generated_lyrics = generator(prompt, max_length=100, num_return_sequences=1)

# Display the generated lyrics
print(generated_lyrics[0]['generated_text'])


Device set to use cpu
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


Started from the bottom, now we're here
And we've been here before
And he never should've left
It's just not your time
You're still here, still in control of it, still with it, now I'm here And I know you could go to bed this morning But you could sleep better that you're gone
So if you didn't wake up this morning
I don't know what I'm going through
I just don't know what to say



In [15]:
trainer.train()
# Check loss every 100 steps


Step,Training Loss
100,2.774100
200,2.735800
300,2.559700
400,2.537200
500,2.326900
600,2.300900


TrainOutput(global_step=651, training_loss=2.52590090375159, metrics={'train_runtime': 3802.4943, 'train_samples_per_second': 0.342, 'train_steps_per_second': 0.171, 'total_flos': 84854587392000.0, 'train_loss': 2.52590090375159, 'epoch': 3.0})